# Distributed Koopman Networks (DKN)

Each neuron is a small Koopman operator: `lift -> K @ z -> project`.  
Many small K matrices in layers replace one monolithic operator.

**Runtime**: Set to **A100 GPU** via Runtime > Change runtime type.

In [ ]:
# 1. Clone and install
!git clone https://github.com/yossideutsch1973/orbit.git
%cd orbit/dkn
!pip install -q torch gymnasium pyyaml numpy pytest

In [ ]:
# 2. Verify GPU and run tests
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
!PYTHONPATH=src python -m pytest tests/ -x -q

In [ ]:
# 3. Run CartPole experiment (DKN vs MLP, 200k steps)
!PYTHONPATH=src python -m experiments.run --config cartpole --device cuda

In [ ]:
# 4. Run LunarLander experiment (DKN vs MLP, 1M steps)
!PYTHONPATH=src python -m experiments.run --config lunarlander --device cuda

In [ ]:
# 5. Custom run — tweak and re-run
import sys
sys.path.insert(0, 'src')

from dkn.network import KoopmanNet, DKNConfig
from dkn.analysis import eigenvalue_summary
from training.runner import train_ppo, count_parameters
from training.ppo import PPOConfig

device = torch.device('cuda')

cfg = DKNConfig(
    state_dim=4, action_dim=2,
    n_layers=3, neurons_per_layer=4,
    d_lift=16, d_out_per_neuron=8, head_hidden=32,
)
net = KoopmanNet(cfg, device)
print(f'Params: {count_parameters(net)}')

result = train_ppo(
    network=net, env_name='CartPole-v1',
    total_steps=200_000, steps_per_update=2048,
    ppo_cfg=PPOConfig(lr=3e-4),
    gamma=0.99, gae_lambda=0.95,
    device=device, seed=42, log_interval=20, is_dkn=True,
)
print(f'Final: {result.final_reward_mean:.1f} +/- {result.final_reward_std:.1f}')
print(eigenvalue_summary(net))